In [3]:
import os  
import gc  
import random  
import warnings  
from pathlib import Path  

import numpy as np  
import pandas as pd  

import matplotlib.pyplot as plt  
import seaborn as sns 

from sklearn.impute import SimpleImputer  
from sklearn.preprocessing import RobustScaler  
from sklearn.preprocessing import QuantileTransformer 
from sklearn.preprocessing import PowerTransformer  
from sklearn.preprocessing import LabelEncoder  
from sklearn.decomposition import PCA  
from sklearn.manifold import TSNE  
from sklearn.model_selection import StratifiedKFold 
from sklearn.model_selection import train_test_split  
from sklearn.metrics import f1_score 
from sklearn.metrics import confusion_matrix  
from sklearn.metrics import classification_report  
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import ExtraTreesClassifier

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import StackingClassifier

from scipy.stats import ks_2samp

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

sns.set_style("darkgrid")

ModuleNotFoundError: No module named 'pandas'

In [2]:
import sys
print(sys.executable)

C:\Users\CHINMAY KAPILA\miniconda3\envs\myenv\python.exe


In [ ]:
train = pd.read_csv("/train.csv")

test = pd.read_csv("/test.csv")

sample_submission = pd.read_csv("/sample_submission.csv")

feature_map = pd.read_csv("/feature_map.csv")

label_map = pd.read_csv("/label_map.csv")

print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

display(train.head())

display(feature_map.head())

themed_to_original = dict(zip(feature_map["themed"], feature_map["original"]))

train_renamed = train.rename(columns=themed_to_original)
test_renamed = test.rename(columns=themed_to_original)

print("Decoded Feature Names Loaded Successfully")

In [ ]:
print(train.info())

missing = train.isnull().mean().sort_values(ascending=False)

plt.figure(figsize=(14,6))
missing.head(30).plot(kind="bar")
plt.title("Top Missing Features")
plt.ylabel("Missing Ratio")
plt.show()

.
plt.figure(figsize=(10,5))
sns.countplot(x="Label", data=train)

plt.title("Target Class Distribution")
plt.show()

sample_cols = train.drop(columns=["id","Label"]).columns[:25]

corr = train[sample_cols].corr()

plt.figure(figsize=(16,12))
sns.heatmap(corr, cmap="coolwarm")
plt.title("Feature Correlation Heatmap")
plt.show()

numeric_cols = train.drop(columns=["id","Label"]).columns[:6]

for col in numeric_cols:

    plt.figure(figsize=(12,4))

    sns.histplot(train[col], bins=50, kde=True)

    plt.title(f"Distribution of {col}")

    plt.show()

X_sample = train.drop(columns=["id","Label"]).sample(3000, random_state=SEED)

y_sample = train.loc[X_sample.index, "Label"]

X_sample = X_sample.fillna(X_sample.median())

scaler = RobustScaler()

X_scaled = scaler.fit_transform(X_sample)

pca = PCA(n_components=2, random_state=SEED)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10,8))

scatter = plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=y_sample,
    cmap="tab10",
    alpha=0.7
)

plt.title("PCA Projection of Network Flows")
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.show()

In [ ]:
train = train.replace([np.inf, -np.inf], np.nan)

test = test.replace([np.inf, -np.inf], np.nan)

duplicates = train.duplicated().sum()

print("Duplicate Rows:", duplicates)

feature_cols = [c for c in train.columns if c not in ["id","Label"]]

for col in feature_cols:

    # Compute robust percentile thresholds.
    lower = train[col].quantile(0.001)
    upper = train[col].quantile(0.999)

    # Clip train.
    train[col] = train[col].clip(lower, upper)

    # Clip test using train statistics to avoid leakage.
    test[col] = test[col].clip(lower, upper)

print("Robust Outlier Clipping Complete")

# ============================================================
# 5. ADVANCED FEATURE ENGINEERING
# ============================================================

def add_engineered_features(df):

    df = df.copy()

    numeric_cols = [c for c in df.columns if c not in ["id","Label"]]

    # Row-wise statistics capture anomalous traffic behavior.
    df["row_mean"] = df[numeric_cols].mean(axis=1)

    df["row_std"] = df[numeric_cols].std(axis=1)

    df["row_max"] = df[numeric_cols].max(axis=1)

    df["row_min"] = df[numeric_cols].min(axis=1)

    # Entropy-like dispersion feature.
    df["row_skew_proxy"] = (
        df["row_mean"] / (df["row_std"] + 1e-6)
    )

    # Log transforms stabilize heavy tails.
    for col in numeric_cols[:15]:

        df[f"{col}_log1p"] = np.log1p(np.abs(df[col]))

    # Ratio features mimic packet/byte relationships.
    if len(numeric_cols) >= 2:

        df["ratio_feature_1"] = (
            df[numeric_cols[0]] /
            (df[numeric_cols[1]] + 1e-3)
        )

    # Quantile bins improve robustness to port obfuscation.
    if "SHIELD_DestinationPort" in df.columns:

        df["port_bin"] = pd.qcut(
            df["SHIELD_DestinationPort"],
            q=10,
            duplicates="drop",
            labels=False
        )

    return df

# Apply engineering.
train = add_engineered_features(train)

test = add_engineered_features(test)

print("Feature Engineering Complete")


In [ ]:

# ============================================================
# 6. DISTRIBUTION SHIFT ANALYSIS
# ============================================================

# Create drift labels.
train_adv = train.drop(columns=["Label"]).copy()

test_adv = test.copy()

train_adv["is_test"] = 0

test_adv["is_test"] = 1

combined = pd.concat([train_adv, test_adv], axis=0)

# Fill missing values.
combined = combined.fillna(combined.median())

X_adv = combined.drop(columns=["id","is_test"])

y_adv = combined["is_test"]

# Train adversarial validator.
adv_model = LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=SEED
)

adv_model.fit(X_adv, y_adv)

# Feature importances reveal unstable features.
importance_df = pd.DataFrame({
    "feature": X_adv.columns,
    "importance": adv_model.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(12,10))

sns.barplot(
    data=importance_df.head(20),
    y="feature",
    x="importance"
)

plt.title("Top Distribution Shift Features")
plt.show()

print("""
Interpretation:
Features with very high adversarial importance differ strongly
between train and test distributions.

Such features may overfit the public leaderboard
and fail on the private leaderboard.
""")

# KS-test drift scores.
drift_scores = []

for col in X_adv.columns[:40]:

    train_vals = train[col].dropna()

    test_vals = test[col].dropna()

    stat, pvalue = ks_2samp(train_vals, test_vals)

    drift_scores.append([col, stat, pvalue])

drift_df = pd.DataFrame(
    drift_scores,
    columns=["feature","ks_stat","pvalue"]
).sort_values("ks_stat", ascending=False)

display(drift_df.head(10))


In [ ]:

# ============================================================
# 7. PREPROCESSING PIPELINE
# ============================================================

target = train["Label"]

X = train.drop(columns=["Label"])

X_test = test.copy()

# Remove IDs from training.
X = X.drop(columns=["id"])

X_test_ids = X_test["id"]

X_test = X_test.drop(columns=["id"])

# Robust imputation.
imputer = SimpleImputer(strategy="median")

X = pd.DataFrame(
    imputer.fit_transform(X),
    columns=X.columns
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns
)

# Robust scaling reduces sensitivity to extreme values.
scaler = RobustScaler()

X = scaler.fit_transform(X)

X_test = scaler.transform(X_test)

# ============================================================
# 8. VALIDATION STRATEGY
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

# ============================================================
# 9. BASELINE MODELS
# ============================================================

baseline_models = {
    "logreg": LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=SEED
    ),

    "rf": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=SEED,
        n_jobs=-1
    ),

    "et": ExtraTreesClassifier(
        n_estimators=400,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    )
}

baseline_scores = {}

for name, model in baseline_models.items():

    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, target)):

        X_train, X_valid = X[tr_idx], X[val_idx]

        y_train, y_valid = target.iloc[tr_idx], target.iloc[val_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)

        score = f1_score(
            y_valid,
            preds,
            average="macro"
        )

        fold_scores.append(score)

        print(f"{name} Fold {fold+1} Macro F1: {score:.5f}")

    baseline_scores[name] = np.mean(fold_scores)

print("\\nBaseline Results")
print(baseline_scores)

# ============================================================
# 10. ADVANCED BOOSTING MODELS
# ============================================================

lgbm = LGBMClassifier(
    objective="multiclass",
    n_estimators=1200,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=SEED,
    class_weight="balanced"
)

xgb = XGBClassifier(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="mlogloss",
    random_state=SEED
)

cat = CatBoostClassifier(
    iterations=1000,
    learning_rate=0.03,
    depth=8,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    verbose=100,
    random_seed=SEED
)

boosting_models = {
    "lightgbm": lgbm,
    "xgboost": xgb,
    "catboost": cat
}

oof_preds = {}

for name, model in boosting_models.items():

    scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, target)):

        X_train, X_valid = X[tr_idx], X[val_idx]

        y_train, y_valid = target.iloc[tr_idx], target.iloc[val_idx]

        model.fit(X_train, y_train)

        preds = model.predict(X_valid)

        score = f1_score(
            y_valid,
            preds,
            average="macro"
        )

        scores.append(score)

        print(f"{name} Fold {fold+1}: {score:.5f}")

    print(f"{name} Mean CV: {np.mean(scores):.5f}")


In [ ]:

# ============================================================
# 11. NOISE AUGMENTATION
# ============================================================

def inject_noise(X, noise_level=0.01):

    X_noisy = X.copy()

    noise = np.random.normal(
        0,
        noise_level,
        X_noisy.shape
    )

    X_noisy += noise

    return X_noisy

# Noise injection improves robustness under hidden drift.
X_augmented = inject_noise(X)

print("Noise Augmentation Complete")

# ============================================================
# 12. HYPERPARAMETER TUNING
# ============================================================

print("""
For runtime efficiency in hackathons:

- Use Optuna with limited trials
- Tune Macro F1 only
- Avoid overfitting public leaderboard
- Prefer conservative regularization
""")

# ============================================================
# 13. ENSEMBLE LEARNING
# ============================================================

ensemble = VotingClassifier(
    estimators=[
        ("lgbm", lgbm),
        ("xgb", xgb),
        ("cat", cat)
    ],
    voting="soft"
)

ensemble.fit(X, target)

print("Ensemble Training Complete")

# ============================================================
# 14. FEATURE IMPORTANCE ANALYSIS
# ============================================================

importance_df = pd.DataFrame({
    "feature": train.drop(columns=["id","Label"]).columns[:len(lgbm.feature_importances_)],
    "importance": lgbm.feature_importances_
}).sort_values("importance", ascending=False)

plt.figure(figsize=(12,10))

sns.barplot(
    data=importance_df.head(25),
    x="importance",
    y="feature"
)

plt.title("LightGBM Feature Importance")
plt.show()

print("""
Cybersecurity Interpretation:
Packet statistics, byte-rate features,
timing features, and TCP behavior indicators
typically dominate intrusion detection performance.
""")

# ============================================================
# 15. FINAL TRAINING
# ============================================================

ensemble.fit(X, target)

final_preds = ensemble.predict(X_test)

# ============================================================
# 16. SUBMISSION CREATION
# ============================================================

submission = pd.DataFrame({
    "id": X_test_ids,
    "Label": final_preds
})

submission.to_csv("submission.csv", index=False)

print(submission.head())

print("submission.csv saved successfully")

# ============================================================
# 17. ERROR ANALYSIS
# ============================================================

print("""
Common Failure Modes:

- BENIGN vs DoS overlap
- PortScan confusion under port obfuscation
- Minority-class instability
- Shift-sensitive packet-rate features

Robust ensembles help reduce variance.
""")

# ============================================================
# 18. FINAL CONCLUSIONS
# ============================================================

print("""
Key Takeaways:

1. Distribution shift awareness is critical.
2. Robust preprocessing matters more than aggressive tuning.
3. Ensembles improve generalization.
4. Macro F1 requires protecting minority classes.
5. Adversarial validation is extremely valuable.

Future Improvements:

- Pseudo labeling
- Calibrated probabilities
- SHAP stability ranking
- Drift-aware weighting
- Autoencoder anomaly features
- TabNet / FT-Transformer experiments
""")
